# DBT Dashboard

Notebook dashboard pour explorer rapidement les marts dbt maintenant que les donnees sont remises d'aplomb.

Vues principales:
- KPIs de synthese
- TVL et prix ETH
- detail TVL: composition token0/token1, active share, out-of-range share
- distribution TVL/liquidity par bins pour le dernier bloc
- distribution TVL/liquidity par bins pour un bloc cible
- TVL active vs out-of-range et APR
- volume, fees et swaps
- activite par tick
- positions ouvertes actuelles


## Prerequis

Si besoin, decommenter et executer:

```python
%pip install pandas plotly clickhouse-connect jupyterlab
```


In [48]:
import os
from textwrap import dedent

import clickhouse_connect

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pandas is required. Run: %pip install pandas plotly clickhouse-connect jupyterlab"
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "plotly is required. Run: %pip install plotly"
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


In [49]:
# Defaults alignes avec dbt/profiles.yml, avec surcharge possible par variables d'environnement.
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8124"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "1234")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "analytics")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f"Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}")


Connected to ClickHouse database=analytics on localhost:8123


In [50]:
def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    for column in ["date", "block_time", "start_time", "end_time", "deployment_ts"]:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    print(f"{len(df):,} rows")
    return df


def build_filters(
    pool: str | None = None,
    date_from: str | None = None,
    date_to: str | None = None,
    table_alias: str | None = None,
):
    clauses = []
    params = {}
    prefix = f"{table_alias}." if table_alias else ""

    if pool:
        clauses.append(f"{prefix}pool = %(pool)s")
        params["pool"] = pool

    if date_from:
        clauses.append(f"{prefix}date >= toDate(%(date_from)s)")
        params["date_from"] = date_from

    if date_to:
        clauses.append(f"{prefix}date <= toDate(%(date_to)s)")
        params["date_to"] = date_to

    where_sql = ""
    if clauses:
        where_sql = "WHERE " + " AND ".join(clauses)
    return where_sql, params


def preview(model_name: str, limit: int = 20) -> pd.DataFrame:
    return query_df(f"SELECT * FROM {model_name} LIMIT {int(limit)}")


def build_tvl_distribution_query(target_block: int) -> str:
    target_block = int(target_block)
    return dedent(
        f"""
        WITH
        pool_state AS (
            SELECT
                s.pool,
                tupleElement(s.state_tuple, 1) AS current_tick,
                sqrt(1e12 / toFloat64(tupleElement(s.state_tuple, 2))) AS sqrt_price_current,
                toFloat64(tupleElement(s.state_tuple, 2)) AS eth_price_latest,
                p.tick_spacing,
                p.token0_symbol,
                p.token1_symbol,
                p.token0_decimals,
                p.token1_decimals
            FROM (
                SELECT
                    pool,
                    argMax(tuple(tick, price_token1_in_token0), tuple(block_number, log_index)) AS state_tuple
                FROM stg_swaps
                WHERE block_number <= {target_block}
                GROUP BY pool
            ) s
            INNER JOIN stg_pools p ON s.pool = p.pool
        ),
        tick_net AS (
            SELECT
                pool,
                tick,
                sum(delta) AS liquidity_net
            FROM (
                SELECT tick_lower AS tick, liquidity_delta AS delta, pool
                FROM int_lp_events
                WHERE block_number <= {target_block}
                UNION ALL
                SELECT tick_upper AS tick, -liquidity_delta AS delta, pool
                FROM int_lp_events
                WHERE block_number <= {target_block}
            )
            GROUP BY pool, tick
            HAVING liquidity_net != 0
        ),
        tick_ranges AS (
            SELECT
                pool,
                tick AS tick_lower,
                lead(tick) OVER (PARTITION BY pool ORDER BY tick) AS tick_upper,
                toFloat64(greatest(
                    sum(liquidity_net) OVER (PARTITION BY pool ORDER BY tick ROWS UNBOUNDED PRECEDING),
                    0
                )) AS liquidity
            FROM tick_net
        ),
        bin_bounds AS (
            SELECT
                r.pool,
                r.tick_lower,
                r.tick_upper,
                r.liquidity,
                ps.tick_spacing,
                ps.token0_symbol,
                ps.token1_symbol,
                ps.token0_decimals,
                ps.token1_decimals,
                pow(1.0001, r.tick_lower / 2.0) AS sp_low,
                pow(1.0001, r.tick_upper / 2.0) AS sp_high,
                greatest(
                    least(ps.sqrt_price_current, pow(1.0001, r.tick_upper / 2.0)),
                    pow(1.0001, r.tick_lower / 2.0)
                ) AS sp,
                ps.current_tick,
                ps.sqrt_price_current,
                ps.eth_price_latest,
                r.tick_lower <= ps.current_tick AND ps.current_tick < r.tick_upper AS is_active_bin
            FROM tick_ranges r
            INNER JOIN pool_state ps ON r.pool = ps.pool
            WHERE r.tick_upper IS NOT NULL
              AND r.liquidity > 0
        ),
        bin_amounts AS (
            SELECT
                pool,
                tick_lower,
                tick_upper,
                liquidity,
                tick_spacing,
                token0_symbol,
                token1_symbol,
                token0_decimals,
                token1_decimals,
                current_tick,
                sqrt_price_current,
                eth_price_latest,
                sp_low,
                sp_high,
                sp,
                is_active_bin,
                liquidity * (sp_high - sp) / (sp * sp_high) / pow(10, token0_decimals) AS amount0,
                liquidity * (sp - sp_low) / pow(10, token1_decimals) AS amount1
            FROM bin_bounds
        )
        SELECT
            pool,
            tick_lower,
            tick_upper,
            tick_spacing,
            liquidity,
            is_active_bin,
            current_tick,
            eth_price_latest,
            token0_symbol,
            token1_symbol,
            amount0,
            amount1,
            amount0 + amount1 * toFloat64(eth_price_latest) AS tvl_usd,
            100.0 * liquidity / sum(liquidity) OVER (PARTITION BY pool) AS pct_liquidity,
            round(1e12 / pow(1.0001, tick_lower), 2) AS price_lower_usdc,
            round(1e12 / pow(1.0001, tick_upper), 2) AS price_upper_usdc,
            (tick_lower + tick_upper) / 2.0 AS tick_mid
        FROM bin_amounts
        WHERE pool = %(pool)s
        ORDER BY price_lower_usdc DESC
        """
    )


def prepare_distribution_window(
    distribution_df: pd.DataFrame,
    width_pct: float = 0.06,
    max_bins: int | None = 300,
):
    if distribution_df.empty:
        return distribution_df.copy(), float("nan"), float("nan"), float("nan")

    df = distribution_df.dropna(subset=["price_lower_usdc", "price_upper_usdc", "eth_price_latest"]).copy()
    df["price_mid_usdc"] = (df["price_lower_usdc"] * df["price_upper_usdc"]).pow(0.5)
    df["bar_width_usdc"] = (df["price_upper_usdc"] - df["price_lower_usdc"]).abs()

    current_price = float(df["eth_price_latest"].iloc[0])
    half_width_pct = width_pct / 2.0
    low = current_price * (1 - half_width_pct)
    high = current_price * (1 + half_width_pct)

    window_df = df[(df["price_upper_usdc"] >= low) & (df["price_lower_usdc"] <= high)].copy()
    window_df = window_df.sort_values("price_mid_usdc")

    if max_bins and len(window_df) > max_bins:
        window_df["distance_to_price"] = (window_df["price_mid_usdc"] - current_price).abs()
        window_df = window_df.nsmallest(max_bins, "distance_to_price").sort_values("price_mid_usdc")
        window_df = window_df.drop(columns=["distance_to_price"])

    return window_df, current_price, low, high


def plot_distribution(
    distribution_df: pd.DataFrame,
    title: str,
    metric: str = "tvl_usd",
    width_pct: float = 0.06,
    max_bins: int | None = 300,
) -> pd.DataFrame:
    if distribution_df.empty:
        print("No distribution data for the selected inputs.")
        return distribution_df

    metric_label = {
        "tvl_usd": "TVL USD",
        "liquidity": "Liquidity",
    }.get(metric, metric)

    window_df, current_price, low, high = prepare_distribution_window(
        distribution_df,
        width_pct=width_pct,
        max_bins=max_bins,
    )

    if window_df.empty:
        print("No bins found inside the selected price window.")
        return window_df

    customdata = window_df[["price_lower_usdc", "price_upper_usdc", "tvl_usd", "liquidity", "pct_liquidity"]].values
    colors = window_df["is_active_bin"].fillna(0).map({1: "#0b6e4f", 0: "#b7d3c6"})

    fig = go.Figure(
        go.Bar(
            x=window_df["price_mid_usdc"],
            y=window_df[metric],
            width=window_df["bar_width_usdc"],
            marker_color=colors,
            customdata=customdata,
            name=metric_label,
            hovertemplate=(
                "Price band: %{customdata[0]:,.2f} - %{customdata[1]:,.2f}<br>"
                "TVL USD: %{customdata[2]:,.2f}<br>"
                "Liquidity: %{customdata[3]:,.0f}<br>"
                "Pct liquidity: %{customdata[4]:.2f}%<extra></extra>"
            ),
        )
    )
    fig.add_vrect(
        x0=low,
        x1=high,
        fillcolor="rgba(33, 150, 243, 0.08)",
        line_width=0,
        annotation_text=f"Window {width_pct:.1%}",
        annotation_position="top left",
    )
    fig.add_vline(
        x=current_price,
        line_width=2,
        line_dash="dash",
        line_color="#111111",
        annotation_text=f"Current price {current_price:,.2f}",
        annotation_position="top",
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        bargap=0,
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="ETH price (USDC)")
    fig.update_yaxes(title_text=metric_label)
    fig.show()

    return window_df[[
        "tick_lower",
        "tick_upper",
        "price_lower_usdc",
        "price_upper_usdc",
        "price_mid_usdc",
        "liquidity",
        "tvl_usd",
        "pct_liquidity",
        "is_active_bin",
    ]]


In [51]:
available_pools = query_df(
    dedent(
        """
        SELECT
            pool,
            min(date) AS min_date,
            max(date) AS max_date,
            count() AS day_count,
            max(tvl_token0_equiv) AS max_tvl_usd
        FROM mart_tvl_daily
        GROUP BY pool
        ORDER BY max_tvl_usd DESC, pool
        """
    )
)
available_pools


1 rows


,pool,min_date,max_date,day_count,max_tvl_usd
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-05,2026-03-13,1774,2.810521e+08


In [52]:
# Ajuste ces parametres pour explorer un autre pool ou une autre fenetre temporelle.
POOL = available_pools.iloc[0]["pool"] if not available_pools.empty else None
DATE_FROM = None
DATE_TO = None
TICK_POINT_LIMIT = 5000
OPEN_POSITIONS_LIMIT = 25
TVL_DISTRIBUTION_WIDTH_PCT = 0.3
TVL_DISTRIBUTION_MAX_BINS = 250
TVL_DISTRIBUTION_METRIC = "tvl_usd"  # or: "liquidity"

latest_block_df = query_df(
    "SELECT max(block_number) AS latest_block FROM stg_swaps WHERE pool = %(pool)s",
    {"pool": POOL},
) if POOL else pd.DataFrame()
LATEST_BLOCK = int(latest_block_df.iloc[0]["latest_block"]) if not latest_block_df.empty else None
TARGET_BLOCK = None  # ex: LATEST_BLOCK - 100000

print("POOL:", POOL)
print("DATE_FROM:", DATE_FROM)
print("DATE_TO:", DATE_TO)
print("LATEST_BLOCK:", LATEST_BLOCK)
print("TARGET_BLOCK:", TARGET_BLOCK)
print("TVL_DISTRIBUTION_WIDTH_PCT:", TVL_DISTRIBUTION_WIDTH_PCT)
print("TVL_DISTRIBUTION_METRIC:", TVL_DISTRIBUTION_METRIC)


1 rows
POOL: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640
DATE_FROM: None
DATE_TO: None
LATEST_BLOCK: 24649412
TARGET_BLOCK: None
TVL_DISTRIBUTION_WIDTH_PCT: 0.3
TVL_DISTRIBUTION_METRIC: tvl_usd


In [53]:
filters, params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)

kpi_df = query_df(
    dedent(
        f"""
        WITH latest_tvl AS (
            SELECT date, tvl_token0_equiv AS tvl_usd, price_token1_in_token0 AS eth_price_close
            FROM mart_tvl_daily
            {filters}
            ORDER BY date DESC
            LIMIT 1
        ),
        latest_fee AS (
            SELECT date, tvl_active_token0 AS tvl_active_usd, tvl_outofrange_token0 AS tvl_outofrange_usd, apr
            FROM mart_fee_yield_daily
            {filters}
            ORDER BY date DESC
            LIMIT 1
        ),
        volume_window AS (
            SELECT
                sum(volume_token0_equiv) AS window_volume_usdc_equiv,
                sum(fees_token0_equiv) AS window_fees_usdc_equiv,
                sum(swap_count) AS window_swap_count
            FROM mart_volume_daily
            {filters}
        )
        SELECT
            lt.date AS latest_date,
            lt.tvl_usd AS latest_tvl_usd,
            lf.tvl_active_usd AS latest_active_tvl_usd,
            lf.tvl_outofrange_usd AS latest_outofrange_tvl_usd,
            lt.eth_price_close AS latest_eth_price_close,
            lf.apr AS latest_apr,
            vw.window_volume_usdc_equiv,
            vw.window_fees_usdc_equiv,
            vw.window_swap_count
        FROM latest_tvl lt
        LEFT JOIN latest_fee lf ON lt.date = lf.date
        CROSS JOIN volume_window vw
        """
    ),
    params,
)

kpi_df


1 rows


,latest_date,latest_tvl_usd,latest_active_tvl_usd,latest_outofrange_tvl_usd,latest_eth_price_close,latest_apr,window_volume_usdc_equiv,window_fees_usdc,window_swap_count
0,2026-03-13,8.746001e+07,6.723836e+07,2.022166e+07,2163.304930,0.504167,5.884806e+11,2.942403e+08,10645912


In [54]:
if not kpi_df.empty:
    row = kpi_df.iloc[0]
    fig = make_subplots(
        rows=2,
        cols=3,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        ],
    )
    indicators = [
        ("Latest TVL USD", row["latest_tvl_usd"]),
        ("Latest Active TVL USD", row["latest_active_tvl_usd"]),
        ("Latest ETH Price", row["latest_eth_price_close"]),
        ("Window Volume USDC Equiv", row["window_volume_usdc_equiv"]),
        ("Window Fees USDC Equiv", row["window_fees_usdc_equiv"]),
        ("Window Swap Count", row["window_swap_count"]),
    ]
    positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]
    for (title, value), (r, c) in zip(indicators, positions):
        fig.add_trace(
            go.Indicator(mode="number", value=float(value or 0), title={"text": title}, number={"valueformat": ",.2f"}),
            row=r,
            col=c,
        )
    fig.update_layout(height=500, title=f"Dashboard KPIs - {POOL}", template="plotly_white")
    fig.show()


In [55]:
filters, params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)

tvl_df = query_df(
    dedent(
        f"""
        SELECT
            date,
            tvl_token0_equiv AS tvl_usd,
            tvl_token0 AS tvl_usdc,
            tvl_token1 AS tvl_weth,
            price_token1_in_token0 AS eth_price_close
        FROM mart_tvl_daily
        {filters}
        ORDER BY date
        """
    ),
    params,
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=tvl_df["date"], y=tvl_df["tvl_usd"], name="TVL USD", mode="lines"), secondary_y=False)
fig.add_trace(go.Scatter(x=tvl_df["date"], y=tvl_df["eth_price_close"], name="ETH Price Close", mode="lines"), secondary_y=True)
fig.update_layout(title=f"TVL and ETH Close Price - {POOL}", template="plotly_white", hovermode="x unified")
fig.update_yaxes(title_text="TVL USD", secondary_y=False)
fig.update_yaxes(title_text="ETH Price (USDC)", secondary_y=True)
fig.show()

tvl_df.tail()


1,774 rows


,date,tvl_usd,tvl_usdc,tvl_weth,eth_price_close
1769,2026-03-09,8.427650e+07,3.862471e+07,22893.111364,1994.127805
1770,2026-03-10,8.539681e+07,4.549963e+07,19591.800459,2036.422091
1771,2026-03-11,8.588936e+07,4.772318e+07,18603.926802,2051.511981
1772,2026-03-12,8.650679e+07,5.096134e+07,17143.505229,2073.405989
1773,2026-03-13,8.746001e+07,6.245280e+07,11559.727476,2163.304930


In [56]:
filters, params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO, table_alias="t")

tvl_detail_df = query_df(
    dedent(
        f"""
        SELECT
            t.date AS date,
            t.tvl_token0_equiv AS tvl_usd,
            t.tvl_token0 AS tvl_usdc,
            t.tvl_token1 AS tvl_weth,
            t.price_token1_in_token0 AS eth_price_close,
            t.tvl_token1 * t.price_token1_in_token0 AS tvl_weth_usd,
            f.tvl_active_token0 AS tvl_active_usd,
            f.tvl_outofrange_token0 AS tvl_outofrange_usd,
            if(t.tvl_token0_equiv = 0, NULL, f.tvl_active_token0 / t.tvl_token0_equiv) AS active_share,
            if(t.tvl_token0_equiv = 0, NULL, f.tvl_outofrange_token0 / t.tvl_token0_equiv) AS outofrange_share,
            f.apr AS apr
        FROM mart_tvl_daily t
        LEFT JOIN mart_fee_yield_daily f
          ON t.pool = f.pool
         AND t.date = f.date
        {filters}
        ORDER BY t.date
        """
    ),
    params,
)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, specs=[[{"secondary_y": False}], [{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=tvl_detail_df["date"], y=tvl_detail_df["tvl_usdc"], stackgroup="token_mix", name="TVL token0 / USDC", mode="lines"), row=1, col=1)
fig.add_trace(go.Scatter(x=tvl_detail_df["date"], y=tvl_detail_df["tvl_weth_usd"], stackgroup="token_mix", name="TVL token1 / WETH in USD", mode="lines"), row=1, col=1)
fig.add_trace(go.Scatter(x=tvl_detail_df["date"], y=tvl_detail_df["active_share"], name="Active share", mode="lines"), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=tvl_detail_df["date"], y=tvl_detail_df["outofrange_share"], name="Out-of-range share", mode="lines"), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=tvl_detail_df["date"], y=tvl_detail_df["apr"], name="APR", mode="lines"), row=2, col=1, secondary_y=True)
fig.update_layout(title=f"TVL detail - composition and active share - {POOL}", template="plotly_white", hovermode="x unified", height=800)
fig.update_yaxes(title_text="TVL USD", row=1, col=1)
fig.update_yaxes(title_text="Share of TVL", tickformat=".0%", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="APR", row=2, col=1, secondary_y=True)
fig.show()

tvl_detail_df.tail()


1,774 rows


,date,tvl_usd,tvl_usdc,tvl_weth,eth_price_close,tvl_weth_usd,tvl_active_usd,tvl_outofrange_usd,active_share,outofrange_share,apr
1769,2026-03-09,8.427650e+07,3.862471e+07,22893.111364,1994.127805,4.565179e+07,7.536752e+07,8.908980e+06,0.894289,0.105711,0.781064
1770,2026-03-10,8.539681e+07,4.549963e+07,19591.800459,2036.422091,3.989718e+07,6.664556e+07,1.875125e+07,0.780422,0.219578,0.708367
1771,2026-03-11,8.588936e+07,4.772318e+07,18603.926802,2051.511981,3.816618e+07,6.715768e+07,1.873168e+07,0.781909,0.218091,0.559117
1772,2026-03-12,8.650679e+07,5.096134e+07,17143.505229,2073.405989,3.554545e+07,6.785349e+07,1.865329e+07,0.784372,0.215628,0.747257
1773,2026-03-13,8.746001e+07,6.245280e+07,11559.727476,2163.304930,2.500722e+07,6.723836e+07,2.022166e+07,0.768790,0.231210,0.504167


In [57]:
latest_distribution_df = query_df(
    dedent(
        """
        SELECT
            pool,
            tick_lower,
            tick_upper,
            liquidity,
            is_active_bin,
            current_tick,
            price_token1_in_token0 AS eth_price_latest,
            amount0,
            amount1,
            tvl_token0 AS tvl_usd,
            pct_liquidity,
            price_lower AS price_lower_usdc,
            price_upper AS price_upper_usdc,
            tick_mid
        FROM mart_tvl_distribution
        WHERE pool = %(pool)s
        ORDER BY price_lower_usdc DESC
        """
    ),
    {"pool": POOL},
)

latest_distribution_window_df = plot_distribution(
    latest_distribution_df,
    title=f"TVL distribution around current price - latest built block - {POOL}",
    metric=TVL_DISTRIBUTION_METRIC,
    width_pct=TVL_DISTRIBUTION_WIDTH_PCT,
    max_bins=TVL_DISTRIBUTION_MAX_BINS,
)

latest_distribution_window_df.head(20)


1,507 rows


,tick_lower,tick_upper,price_lower_usdc,price_upper_usdc,price_mid_usdc,liquidity,tvl_usd,pct_liquidity,is_active_bin
959,200890,200900,1887.53,1885.65,1886.589766,5.426105e+18,117835.213710,0.147310,0
958,200880,200890,1889.42,1887.53,1888.474764,5.266342e+18,114422.935658,0.142973,0
957,200870,200880,1891.31,1889.42,1890.364764,5.274558e+18,114658.755077,0.143196,0
956,200860,200870,1893.20,1891.31,1892.254764,6.633575e+18,144273.302790,0.180091,0
955,200850,200860,1895.10,1893.20,1894.149762,1.422545e+19,309543.349094,0.386199,0
954,200840,200850,1896.99,1895.10,1896.044765,1.422566e+19,309702.713045,0.386205,0
953,200830,200840,1898.89,1896.99,1897.939762,1.421974e+19,309728.654608,0.386044,0
952,200820,200830,1900.79,1898.89,1899.839762,1.422193e+19,309931.321661,0.386103,0
951,200810,200820,1902.69,1900.79,1901.739763,1.423853e+19,310448.105818,0.386554,0
950,200800,200810,1904.60,1902.69,1903.644760,1.424788e+19,310807.528684,0.386808,0


In [58]:
if TARGET_BLOCK is None:
    print(f"Set TARGET_BLOCK to inspect a historical distribution. Latest block for {POOL}: {LATEST_BLOCK}")
else:
    target_distribution_df = query_df(
        build_tvl_distribution_query(TARGET_BLOCK),
        {"pool": POOL},
    )

    target_distribution_window_df = plot_distribution(
        target_distribution_df,
        title=f"TVL distribution around current price - block {TARGET_BLOCK} - {POOL}",
        metric=TVL_DISTRIBUTION_METRIC,
        width_pct=TVL_DISTRIBUTION_WIDTH_PCT,
        max_bins=TVL_DISTRIBUTION_MAX_BINS,
    )

    target_distribution_window_df.head(20)


Set TARGET_BLOCK to inspect a historical distribution. Latest block for 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640: 24649412


In [59]:
filters, params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)

fee_df = query_df(
    dedent(
        f"""
        SELECT
            date,
            tvl_total_token0 AS tvl_total_usd,
            tvl_active_token0 AS tvl_active_usd,
            tvl_outofrange_token0 AS tvl_outofrange_usd,
            fees_token0_equiv AS fees_usdc_equiv,
            apr
        FROM mart_fee_yield_daily
        {filters}
        ORDER BY date
        """
    ),
    params,
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=fee_df["date"], y=fee_df["tvl_active_usd"], stackgroup="one", name="Active TVL USD", mode="lines"), secondary_y=False)
fig.add_trace(go.Scatter(x=fee_df["date"], y=fee_df["tvl_outofrange_usd"], stackgroup="one", name="Out-of-range TVL USD", mode="lines"), secondary_y=False)
fig.add_trace(go.Scatter(x=fee_df["date"], y=fee_df["apr"], name="APR", mode="lines"), secondary_y=True)
fig.update_layout(title=f"Active vs Out-of-range TVL and APR - {POOL}", template="plotly_white", hovermode="x unified")
fig.update_yaxes(title_text="TVL USD", secondary_y=False)
fig.update_yaxes(title_text="APR", secondary_y=True)
fig.show()

fee_df.tail()


1,774 rows


,date,tvl_total_usd,tvl_active_usd,tvl_outofrange_usd,fees_usdc,apr
1769,2026-03-09,8.427650e+07,7.536752e+07,8.908980e+06,161278.996960,0.781064
1770,2026-03-10,8.539681e+07,6.664556e+07,1.875125e+07,129341.119146,0.708367
1771,2026-03-11,8.588936e+07,6.715768e+07,1.873168e+07,102873.881458,0.559117
1772,2026-03-12,8.650679e+07,6.785349e+07,1.865329e+07,138915.001420,0.747257
1773,2026-03-13,8.746001e+07,6.723836e+07,2.022166e+07,92875.034049,0.504167


In [60]:
filters, params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)

volume_df = query_df(
    dedent(
        f"""
        SELECT
            date,
            volume_token0 AS volume_usdc,
            volume_token1 AS volume_weth,
            volume_token0_equiv AS volume_usdc_equiv,
            swap_count,
            fees_token0,
            fees_token1,
            fees_token0_equiv AS fees_usdc_equiv
        FROM mart_volume_daily
        {filters}
        ORDER BY date
        """
    ),
    params,
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=volume_df["date"], y=volume_df["volume_usdc_equiv"], name="Volume USDC Equiv"), secondary_y=False)
fig.add_trace(go.Scatter(x=volume_df["date"], y=volume_df["swap_count"], name="Swap Count", mode="lines+markers"), secondary_y=True)
fig.update_layout(title=f"Input Notional Volume and Swap Count - {POOL}", template="plotly_white", hovermode="x unified")
fig.update_yaxes(title_text="Volume USDC Equiv", secondary_y=False)
fig.update_yaxes(title_text="Swap Count", secondary_y=True)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=volume_df["date"], y=volume_df["volume_usdc"], name="Volume token0 / USDC side", mode="lines"))
fig.add_trace(go.Scatter(x=volume_df["date"], y=volume_df["volume_weth"], name="Volume token1 / WETH side", mode="lines"))
fig.update_layout(title=f"Raw Token-side Volumes - {POOL}", template="plotly_white", hovermode="x unified")
fig.show()

fig = go.Figure()
fig.add_trace(go.Bar(x=volume_df["date"], y=volume_df["fees_usdc_equiv"], name="Fees USDC Equiv"))
fig.add_trace(go.Scatter(x=volume_df["date"], y=volume_df["fees_token0"], name="Fees token0 / USDC", mode="lines+markers"))
fig.add_trace(go.Scatter(x=volume_df["date"], y=volume_df["fees_token1"], name="Fees token1 / WETH", mode="lines+markers"))
fig.update_layout(title=f"Generated Fees - token-aware - {POOL}", template="plotly_white", hovermode="x unified")
fig.show()

volume_df.tail()


1,774 rows


,date,volume_usdc,volume_weth,volume_usdc_equiv,swap_count,fees_token0,fees_token1,fees_usdc
1769,2026-03-09,3.225357e+08,161264.979437,3.225580e+08,7619,83013.359691,39.125934,161278.996960
1770,2026-03-10,2.586675e+08,126346.627474,2.586822e+08,6543,66409.424940,30.737502,129341.119146
1771,2026-03-11,2.057335e+08,100460.169265,2.057478e+08,5433,51932.386256,24.883720,102873.881458
1772,2026-03-12,2.780891e+08,133620.654408,2.778300e+08,6399,70257.734866,33.065379,138915.001420
1773,2026-03-13,1.857386e+08,86949.933593,1.857501e+08,5107,49409.248355,20.343332,92875.034049


In [61]:
filters, params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)
params["tick_point_limit"] = TICK_POINT_LIMIT

tick_df = query_df(
    dedent(
        f"""
        SELECT
            date,
            tick,
            time_seconds,
            volume_token0_equiv AS volume_usdc,
            swap_count
        FROM mart_tick_daily
        {filters}
        ORDER BY date, volume_usdc DESC
        LIMIT %(tick_point_limit)s
        """
    ),
    params,
)

fig = px.scatter(
    tick_df,
    x="date",
    y="tick",
    size="volume_usdc",
    color="time_seconds",
    hover_data=["swap_count"],
    title=f"Tick Activity Map - {POOL}",
    template="plotly_white",
)
fig.show()

tick_df.head()


5,000 rows


,date,tick,time_seconds,volume_usdc,swap_count
0,2021-05-05,194725,423,2020.724832,1
1,2021-05-05,194648,93,514.279985,1
2,2021-05-05,194645,821,499.756414,1
3,2021-05-05,194641,298,365.000000,1
4,2021-05-05,194643,62,176.180828,1


In [62]:
position_filters = ""
position_params = {}
if POOL:
    position_filters = "WHERE pool = %(pool)s"
    position_params["pool"] = POOL
position_params["limit"] = OPEN_POSITIONS_LIMIT

positions_df = query_df(
    dedent(
        f"""
        SELECT
            owner,
            tick_lower,
            tick_upper,
            net_liquidity,
            is_active,
            position_token0 AS position_usd
        FROM mart_open_positions
        {position_filters}
        ORDER BY position_usd DESC
        LIMIT %(limit)s
        """
    ),
    position_params,
)

fig = px.bar(
    positions_df,
    x="owner",
    y="position_usd",
    color="is_active",
    hover_data=["tick_lower", "tick_upper", "net_liquidity"],
    title=f"Top Open Positions - {POOL}",
    template="plotly_white",
)
fig.update_layout(xaxis_title="Owner", yaxis_title="Position USD")
fig.show()

positions_df.head(20)


25 rows


,owner,tick_lower,tick_upper,net_liquidity,is_active,position_usd
0,0xc36442b4a4522e871399cd717abdd847ab11fe88,199380,200860,7598705807111945268,1,2.673992e+07
1,0xc36442b4a4522e871399cd717abdd847ab11fe88,200150,201940,2794337315720186614,0,1.145849e+07
2,0xc36442b4a4522e871399cd717abdd847ab11fe88,195610,209470,347980996739758993,1,7.917003e+06
3,0xc36442b4a4522e871399cd717abdd847ab11fe88,199310,200870,1352857039838262121,1,4.995915e+06
4,0xc36442b4a4522e871399cd717abdd847ab11fe88,194710,212350,170278993464055372,1,4.039246e+06
5,0xc36442b4a4522e871399cd717abdd847ab11fe88,198320,202640,323586592167242883,1,3.088527e+06
6,0xc36442b4a4522e871399cd717abdd847ab11fe88,191150,214170,7047538453218,1,2.758369e+06
7,0xc36442b4a4522e871399cd717abdd847ab11fe88,192890,200360,108501304409652587,1,1.639785e+06
8,0xc36442b4a4522e871399cd717abdd847ab11fe88,194190,194220,953335078185974,0,1.393907e+06
9,0xc36442b4a4522e871399cd717abdd847ab11fe88,196950,206450,65989735192964238,1,1.278389e+06


In [63]:
# Exploration libre d'un modele materialise.
preview("mart_tvl_daily", limit=10)


10 rows


,pool,date,tvl_usdc,tvl_weth,eth_price_close,tvl_usd
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-05,3.995454e+03,3.378178,3496.203173,1.580625e+04
1,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-06,2.609669e+05,72.192725,3496.292879,5.133738e+05
2,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-07,1.565575e+06,336.937391,3483.701459,2.739365e+06
3,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-08,1.208533e+06,88.299866,3884.370431,1.551523e+06
4,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-09,1.102904e+06,109.361547,3909.453752,1.530448e+06
5,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-10,1.019438e+06,174.605384,3968.366566,1.712336e+06
6,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-11,1.234784e+06,100.227122,4153.478530,1.651075e+06
7,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-12,7.762898e+05,99.554158,3755.544027,1.150170e+06
8,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-13,7.378419e+05,115.792042,3717.384453,1.168285e+06
9,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-14,9.198546e+05,63.875034,4088.482177,1.181007e+06
